### Testing on weather data

In [28]:
import sys
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import pytorch_lightning as pl
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import os
import logging


In [9]:
sys.path.insert(0, '/home/sandeep/DSIPTS_PTF')

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


In [11]:

DATA_PATH = '/home/sandeep/DSIPTS_PTF/data/'
FILE_PATH = DATA_PATH + 'weather.csv'
PAST_LEN = 96  # Use 96 hours (4 days) of data
FUTURE_LEN = 96   # To predict 96 hours (4 days)
BATCH_SIZE = 32
TARGET_COL = 'y'  # Temperature column in weather data

Loading weather Data (same as weather_d1_d2_experiment.py)

In [3]:
DATA_PATH = '/home/sandeep/DSIPTS_PTF/data/'
FILE_PATH = DATA_PATH + 'weather.csv'
PAST_LEN = 96  # Use 96 hours (4 days) of data
FUTURE_LEN = 96   # To predict 96 hours (4 days)
BATCH_SIZE = 32
TARGET_COL = 'y'  # Temperature column in weather data

In [12]:
def create_sequences(data, past_len, future_len, target_col_idx):
    """
    Creates sequences for time series forecasting.
    X = (n_samples, past_len, n_features)
    y = (n_samples, future_len, 1)  (predicting one target feature)
    """
    xs, ys = [], []
    for i in range(len(data) - past_len - future_len + 1):
        x = data[i:(i + past_len), :]
        y = data[(i + past_len):(i + past_len + future_len), target_col_idx]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys).reshape(-1, future_len)

In [14]:
class WeatherTimeSeriesModule(pl.LightningDataModule):
    def __init__(self, data_path: str, past_len: int, future_len: int, target_col: str, batch_size: int = 32):
        super().__init__()
        print(f"\n[TRACE] CALLING WeatherTimeSeriesModule.__init__()")
        
        self.data_path = data_path
        self.past_len = past_len
        self.future_len = future_len
        self.batch_size = batch_size
        self.target_col = target_col
        
        # 1. Initialize scaler ONCE. It is not fitted here.
        self.scaler = StandardScaler()
        self.scaler_fitted = False
        
        # Placeholders for data and splits
        self.raw_data = None
        self.feature_cols = None
        self.target_col_idx = -1
        
        self.train_raw = None
        self.val_raw = None
        self.test_raw = None
        
        self.train_dataset = None
        self.val_dataset = None
        self.test_dataset = None

    def prepare_data(self):
        # Called once per node. Good for downloading/loading data.
        print(f"\n[TRACE] --- CALLING prepare_data() ---")
        if not os.path.exists(self.data_path):
            print(f"Data not found, generating '{self.data_path}'...")
            create_dummy_weather_data(self.data_path)
        else:
            print(f"Found '{self.data_path}'.")
            
        df = pd.read_csv(self.data_path)
        # We only use features, target is derived from 'temperature'
        self.feature_cols = ['temperature', 'humidity', 'pressure']
        self.raw_data = df[self.feature_cols].values
        self.target_col_idx = self.feature_cols.index(self.target_col)
        print(f"[TRACE] --- prepare_data() FINISHED ---")


    def setup(self, stage: str):
        # This is the core logic. Called once per stage ('fit', 'test', 'predict') per process.
        print(f"\n[TRACE] --- CALLING setup(stage='{stage}') ---")
        
        # 🎯 STAGE 1: Split data and fit scaler (ONLY ONCE)
        if not self.scaler_fitted:
            print("[TRACE] Scaler not fitted. This is the first call to setup().")
            print("[TRACE] Performing train/val/test split...")
            
            # Split data (70% train, 15% val, 15% test)
            train_val_raw, self.test_raw = train_test_split(
                self.raw_data, test_size=0.15, shuffle=False
            )
            self.train_raw, self.val_raw = train_test_split(
                train_val_raw, test_size=(0.15/0.85), shuffle=False # 0.15 / 0.85 = ~0.176
            )
            print(f"[TRACE] Split shapes: Train={self.train_raw.shape}, Val={self.val_raw.shape}, Test={self.test_raw.shape}")
            
            # 🎯 STAGE 2: Fit scaler on TRAINING data only
            print("[TRACE] Fitting StandardScaler on TRAINING data...")
            self.scaler.fit(self.train_raw)
            self.scaler_fitted = True
            print("[TRACE] Scaler has been fitted and 'scaler_fitted' flag is set to True.")
        
        else:
            print(f"[TRACE] Scaler already fitted. Skipping split and fit steps.")

        # 🎯 STAGE 3: Transform data and create stage-specific datasets
        if stage == "fit":
            print(f"[TRACE] Setting up 'fit' stage...")
            train_scaled = self.scaler.transform(self.train_raw)
            val_scaled = self.scaler.transform(self.val_raw)
            
            X_train, y_train = create_sequences(train_scaled, self.past_len, self.future_len, self.target_col_idx)
            X_val, y_val = create_sequences(val_scaled, self.past_len, self.future_len, self.target_col_idx)

            self.train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32))
            self.val_dataset = TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32))
            print(f"[TRACE] 'fit' datasets created.")

        if stage == "test":
            print(f"[TRACE] Setting up 'test' stage...")
            test_scaled = self.scaler.transform(self.test_raw)
            X_test, y_test = create_sequences(test_scaled, self.past_len, self.future_len, self.target_col_idx)
            self.test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32))
            print(f"[TRACE] 'test' dataset created.")

        if stage == "predict":
            # (Optional)
            print(f"[TRACE] Setting up 'predict' stage...")
            # Create a predict_dataset, often the same as test
            if self.test_dataset is None:
                 test_scaled = self.scaler.transform(self.test_raw)
                 X_test, y_test = create_sequences(test_scaled, self.past_len, self.future_len, self.target_col_idx)
                 self.test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32))
            self.predict_dataset = self.test_dataset # or any other data
            print(f"[TRACE] 'predict' dataset created.")
        
        print(f"[TRACE] --- setup(stage='{stage}') FINISHED ---")

    def train_dataloader(self):
        print("[TRACE] CALLING train_dataloader()")
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=4)

    def val_dataloader(self):
        print("[TRACE] CALLING val_dataloader()")
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False, num_workers=4)

    def test_dataloader(self):
        print("[TRACE] CALLING test_dataloader()")
        return DataLoader(self.test_dataset, batch_size=self.batch_size, shuffle=False, num_workers=4)
        
    def predict_dataloader(self):
        print("[TRACE] CALLING predict_dataloader()")
        return DataLoader(self.predict_dataset, batch_size=self.batch_size, shuffle=False, num_workers=4)


In [20]:
class SimpleModel(pl.LightningModule):
    """A simple model for testing the DataModule."""
    def __init__(self, input_size, output_size):
        super().__init__()
        print(f"\n[TRACE] CALLING SimpleModel.__init__()")
        # Flatten input (batch_size, past_len, n_features) -> (batch_size, past_len * n_features)
        self.flatten = nn.Flatten()
        self.linear = nn.Linear(input_size, output_size)
        self.criterion = nn.MSELoss()
        
    def forward(self, x):
        x = self.flatten(x)
        return self.linear(x)

    def _common_step(self, batch, stage):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        self.log(f'{stage}_loss', loss, on_epoch=True, prog_bar=True)
        return loss

    def training_step(self, batch, batch_idx):
        return self._common_step(batch, 'train')

    def validation_step(self, batch, batch_idx):
        return self._common_step(batch, 'val')
        
    def test_step(self, batch, batch_idx):
        return self._common_step(batch, 'test')

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=0.001)

Starting lightning datamodule end-2-end

In [21]:
dm = WeatherTimeSeriesModule(
        data_path=FILE_PATH,
        past_len=PAST_LEN,
        future_len=FUTURE_LEN,
        target_col=TARGET_COL,
        batch_size=BATCH_SIZE
    )


[TRACE] CALLING WeatherTimeSeriesModule.__init__()


In [23]:
dm.prepare_data()
n_features = len(dm.feature_cols)


[TRACE] --- CALLING prepare_data() ---
Loading weather data from '/home/sandeep/DSIPTS_PTF/data/weather.csv'...
[TRACE] Data shape: (52696, 21)
[TRACE] Features: ['p (mbar)', 'T (degC)', 'Tpot (K)', 'Tdew (degC)', 'rh (%)', 'VPmax (mbar)', 'VPact (mbar)', 'VPdef (mbar)', 'sh (g/kg)', 'H2OC (mmol/mol)', 'rho (g/m**3)', 'wv (m/s)', 'max. wv (m/s)', 'wd (deg)', 'rain (mm)', 'raining (s)', 'SWDR (W/m�)', 'PAR (�mol/m�/s)', 'max. PAR (�mol/m�/s)', 'Tlog (degC)', 'y']
[TRACE] Target column index: 20
[TRACE] --- prepare_data() FINISHED ---


init model now

In [24]:
input_size = PAST_LEN * n_features
output_size = FUTURE_LEN

model = SimpleModel(input_size, output_size)


[TRACE] CALLING SimpleModel.__init__()


In [ ]:
model = SimpleModel(input_size, output_size)


[TRACE] CALLING SimpleModel.__init__()


ININTIALIZING The trainer

In [26]:
trainer = pl.Trainer(
        max_epochs=2,
        accelerator="auto",
        enable_checkpointing=False,
        logger=False,
        num_sanity_val_steps=0
    )

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


calling fit on the trainer

In [29]:
trainer.fit(model, dm)


[TRACE] --- CALLING prepare_data() ---
Loading weather data from '/home/sandeep/DSIPTS_PTF/data/weather.csv'...
[TRACE] Data shape: (52696, 21)
[TRACE] Features: ['p (mbar)', 'T (degC)', 'Tpot (K)', 'Tdew (degC)', 'rh (%)', 'VPmax (mbar)', 'VPact (mbar)', 'VPdef (mbar)', 'sh (g/kg)', 'H2OC (mmol/mol)', 'rho (g/m**3)', 'wv (m/s)', 'max. wv (m/s)', 'wd (deg)', 'rain (mm)', 'raining (s)', 'SWDR (W/m�)', 'PAR (�mol/m�/s)', 'max. PAR (�mol/m�/s)', 'Tlog (degC)', 'y']
[TRACE] Target column index: 20
[TRACE] --- prepare_data() FINISHED ---

[TRACE] --- CALLING setup(stage='TrainerFn.FITTING') ---
[TRACE] Scaler not fitted. This is the first call to setup().
[TRACE] Performing train/val/test split...
[TRACE] Split shapes: Train=(36886, 21), Val=(7905, 21), Test=(7905, 21)
[TRACE] Fitting StandardScaler on TRAINING data...
[TRACE] Scaler has been fitted and 'scaler_fitted' flag is set to True.
[TRACE] Setting up 'fit' stage...



  | Name      | Type    | Params | Mode 
----------------------------------------------
0 | flatten   | Flatten | 0      | train
1 | linear    | Linear  | 193 K  | train
2 | criterion | MSELoss | 0      | train
----------------------------------------------
193 K     Trainable params
0         Non-trainable params
193 K     Total params
0.775     Total estimated model params size (MB)
3         Modules in train mode
0         Modules in eval mode


[TRACE] 'fit' datasets created.
[TRACE] --- setup(stage='TrainerFn.FITTING') FINISHED ---
[TRACE] CALLING train_dataloader()
[TRACE] CALLING val_dataloader()
Epoch 1: 100%|██████████| 1147/1147 [00:17<00:00, 64.72it/s, train_loss_step=0.183, val_loss=0.104, train_loss_epoch=1.090] 

`Trainer.fit` stopped: `max_epochs=2` reached.


Epoch 1: 100%|██████████| 1147/1147 [00:17<00:00, 64.71it/s, train_loss_step=0.183, val_loss=0.104, train_loss_epoch=1.090]


TEsting the model 

In [30]:
trainer.test(model, dm)


[TRACE] --- CALLING prepare_data() ---
Loading weather data from '/home/sandeep/DSIPTS_PTF/data/weather.csv'...
[TRACE] Data shape: (52696, 21)
[TRACE] Features: ['p (mbar)', 'T (degC)', 'Tpot (K)', 'Tdew (degC)', 'rh (%)', 'VPmax (mbar)', 'VPact (mbar)', 'VPdef (mbar)', 'sh (g/kg)', 'H2OC (mmol/mol)', 'rho (g/m**3)', 'wv (m/s)', 'max. wv (m/s)', 'wd (deg)', 'rain (mm)', 'raining (s)', 'SWDR (W/m�)', 'PAR (�mol/m�/s)', 'max. PAR (�mol/m�/s)', 'Tlog (degC)', 'y']
[TRACE] Target column index: 20
[TRACE] --- prepare_data() FINISHED ---

[TRACE] --- CALLING setup(stage='TrainerFn.TESTING') ---
[TRACE] Scaler already fitted. Skipping split and fit steps.
[TRACE] Setting up 'test' stage...
[TRACE] 'test' dataset created.
[TRACE] --- setup(stage='TrainerFn.TESTING') FINISHED ---
[TRACE] CALLING test_dataloader()
Testing DataLoader 0: 100%|██████████| 242/242 [00:03<00:00, 65.59it/s]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │    0.07394308596849442    │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.07394308596849442}]